#**Important notes**

- this notebook shows the models tuned on 9 features (colors + magnitudes), for our ablation part we are comaparing models tuned to mags alone and colors alone.

    we tuned those models speratly and saved them on the drive. its identical to the code here just diff featrues.


- the code for this notebook was run locally. the models (```.joblibs```) were saved and uploaded to drive for the next notebook

In [ ]:
!jupyter nbconvert --execute --to html "/content/Project_Notebook_3_Tuning.ipynb"

[NbConvertApp] Converting notebook /content/Project_Notebook_3_Tuning.ipynb to html
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/iostream.py", line 563, in write
    self._schedule_flush()
  File "/usr/local/l

loading dataframe from drive

In [ ]:
import pandas as pd
df = pd.read_csv('https://drive.google.com/uc?export=download&id=1k26vJfnpQx-Vl509-pjxhBkWu5efS12s')

##setup

same code from notebook #2

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# define features X and target y
# our features are magnitudes + color
# the target is what we are trying to estimate, redshift.
feature_cols = ['u', 'g', 'r', 'i', 'z', 'u_g', 'g_r', 'r_i', 'i_z']
X = df[feature_cols].values
y = df['redshift'].values

print(f"{X.shape}")
print(f"{y.shape}")

(57241, 9)
(57241,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # 20% for testing
    random_state=42    # reproducibility
)

print(f"Training set: {X_train.shape[0]} galaxies")
print(f"Test set: {X_test.shape[0]} galaxies")

Training set: 45792 galaxies
Test set: 11449 galaxies


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training mean: {X_train_scaled.mean(axis=0).round(3)}")
print(f"Training std:  {X_train_scaled.std(axis=0).round(3)}")

Training mean: [-0.  0. -0. -0.  0. -0. -0.  0. -0.]
Training std:  [1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [ ]:
#eval function

from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(y_true, y_pred, model_name):
    # Standard regression metrics
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)

    # Photo-z specific metrics
    dz = (y_pred - y_true) / (1 + y_true)
    bias = np.mean(dz)
    sigma_nmad = 1.48 * np.median(np.abs(dz))
    outlier_fraction = np.mean(np.abs(dz) > 0.15)

    print(f"\n--- {model_name} ---")
    print(f"MAE:              {mae:.4f}")
    print(f"MSE:              {mse:.4f}")
    print(f"Bias:             {bias:.4f}")
    print(f"σ_NMAD:           {sigma_nmad:.4f}")
    print(f"Outlier fraction: {outlier_fraction:.4%}")

    return {'model': model_name, 'MAE': mae, 'MSE': mse,
            'bias': bias, 'sigma_NMAD': sigma_nmad,
            'outlier_fraction': outlier_fraction}

**note:** tuning results will be discused in the paper

#k-Nearest Neighbors

kNN has 3 hyperparameters we should tune.

```
n_neighbors, weights and p
```

we use use RanomizedSearchCV with 50 iterations and 5 cross valdiation.

this means we randomly sample 50 hyperparameters combos, for each combo we train and eval 5 times using diff trian/valid split of our training set. the combo with the  best avg MSE wins


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import time

knn_grid = {
    'n_neighbors': randint(1, 201),      # 1 to 200 inclusive
    'weights': ['uniform', 'distance'],
    'p': [1, 2],
}

knn_search = RandomizedSearchCV(
    estimator=KNeighborsRegressor(n_jobs=-1),
    param_distributions=knn_grid,
    n_iter=50,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

t0 = time.time()
knn_search.fit(X_train_scaled, y_train)
knn_tune_time = time.time() - t0

print(f"Best params: {knn_search.best_params_}")
print(f"Best CV MSE: {-knn_search.best_score_:.4f}")
print(f"Tuning time: {knn_tune_time:.1f}s")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'n_neighbors': 22, 'p': 1, 'weights': 'distance'}
Best CV MSE: 0.0073
Tuning time: 60.2s


In [ ]:
y_pred_knn_tuned = knn_search.predict(X_test_scaled)
results_knn_tuned = evaluate(y_test, y_pred_knn_tuned, "kNN (tuned)")


--- kNN (tuned) ---
MAE:              0.0477
MSE:              0.0075
Bias:             0.0027
σ_NMAD:           0.0274
Outlier fraction: 2.9086%


#Random Forest

there is 5 hyperparameters that matter for us here.



```
n_estimators
max_features
max_depth
min_samples_split
min_samples_leaf
```

we basically followed Henghes (2021) for numbers but with some adjusments. but we dropped
```
crierion = "absolute_error"
```
its too slow, and we are optimzing for MSE, not MAE.


In [ ]:
# Section 3 — Random Forest tuning
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import time

rf_grid = {
    'n_estimators': randint(50, 301),
    'max_features': [1, 2, 3, 4, 5, 'sqrt', 'log2'],
    'max_depth': [None, 10, 20, 30, 50],
    'min_samples_split': randint(2, 21),
    'min_samples_leaf': randint(1, 11),
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_grid,
    n_iter=50,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

t0 = time.time()
rf_search.fit(X_train_scaled, y_train)
rf_tune_time = time.time() - t0

print(f"Best params: {rf_search.best_params_}")
print(f"Best CV MSE: {-rf_search.best_score_:.4f}")
print(f"Tuning time: {rf_tune_time:.1f}s")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'max_depth': 50, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 13, 'n_estimators': 211}
Best CV MSE: 0.0069
Tuning time: 617.9s


In [ ]:
y_pred_rf_tuned = rf_search.predict(X_test_scaled)
results_rf_tuned = evaluate(y_test, y_pred_rf_tuned, "RF (tuned)")


--- RF (tuned) ---
MAE:              0.0460
MSE:              0.0071
Bias:             0.0023
σ_NMAD:           0.0259
Outlier fraction: 2.8387%


#XGBoost

XGBoost has the most hyperparameters of all our models. 8 that we can tune:

```
n_estimators
max_depth
learning_rate
subsample
colsample_bytree
min_child_weight
reg_alpha
reg_lambda
```



In [ ]:
# Section 4 — XGBoost tuning
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import time

xgb_grid = {
    'n_estimators': randint(100, 1001),
    'max_depth': randint(3, 11),
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': randint(1, 11),
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.5, 1, 2, 5],
}

xgb_search = RandomizedSearchCV(
    estimator=XGBRegressor(
        random_state=42,
        n_jobs=-1,
        tree_method='hist',   # fast histogram-based algorithm
        verbosity=0,
    ),
    param_distributions=xgb_grid,
    n_iter=50,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=1,   # let XGBoost handle parallelism
    verbose=1,
)

t0 = time.time()
xgb_search.fit(X_train_scaled, y_train)
xgb_tune_time = time.time() - t0

print(f"Best params: {xgb_search.best_params_}")
print(f"Best CV MSE: {-xgb_search.best_score_:.4f}")
print(f"Tuning time: {xgb_tune_time:.1f}s")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 10, 'min_child_weight': 1, 'n_estimators': 747, 'reg_alpha': 1, 'reg_lambda': 2, 'subsample': 0.8}
Best CV MSE: 0.0069
Tuning time: 317.8s


In [ ]:
y_pred_xgb_tuned = xgb_search.predict(X_test_scaled)
results_xgb_tuned = evaluate(y_test, y_pred_xgb_tuned, "XGBoost (tuned)")


--- XGBoost (tuned) ---
MAE:              0.0462
MSE:              0.0072
Bias:             0.0024
σ_NMAD:           0.0264
Outlier fraction: 2.8125%


#MLP

for this we need the number of epochs to be high enough to converge, the default is 200 but for our 50k+ dataset its a bit low. 500 is more than enough.

for activaion i have to point out that some papers on this subject use tanh. its useful to test if its better than ReLU here

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV
import time

mlp_grid = {
    'hidden_layer_sizes': [
        (50,), (100,), (200,),
        (50, 50), (100, 100), (200, 100),
        (100, 50, 25), (100, 100, 100),
    ],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'learning_rate_init': [0.0001, 0.001, 0.01],
    'batch_size': [32, 128, 256],
    'solver': ['adam'],
}

mlp_search = RandomizedSearchCV(
    estimator=MLPRegressor(
        max_iter=500,
        random_state=42,
    ),
    param_distributions=mlp_grid,
    n_iter=50,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

t0 = time.time()
mlp_search.fit(X_train_scaled, y_train)
mlp_tune_time = time.time() - t0

print(f"Best params: {mlp_search.best_params_}")
print(f"Best CV MSE: {-mlp_search.best_score_:.4f}")
print(f"Tuning time: {mlp_tune_time:.1f}s")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best params: {'solver': 'adam', 'learning_rate_init': 0.0001, 'hidden_layer_sizes': (100, 50, 25), 'batch_size': 32, 'alpha': 0.001, 'activation': 'relu'}
Best CV MSE: 0.0071
Tuning time: 307.8s


In [ ]:
y_pred_mlp_tuned = mlp_search.predict(X_test_scaled)
results_mlp_tuned = evaluate(y_test, y_pred_mlp_tuned, "MLP (tuned)")


--- MLP (tuned) ---
MAE:              0.0472
MSE:              0.0073
Bias:             0.0036
σ_NMAD:           0.0275
Outlier fraction: 2.7950%




---



---



saving the tuned models + scaler so we don't have to run them again later

In [ ]:
"""

import joblib

joblib.dump(knn_search.best_estimator_, 'knn.joblib')
joblib.dump(rf_search.best_estimator_, 'rf.joblib')
joblib.dump(xgb_search.best_estimator_, 'xgb.joblib')
joblib.dump(mlp_search.best_estimator_, 'mlp.joblib')
joblib.dump(scaler, 'scaler.joblib')

"""

['scaler.joblib']